In [1]:
import os
import sys
import torch




# folder of the notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))  
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "../.."))  # src/resnet18/

# add src/ to path for imports
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
sys.path.append(SRC_DIR)

In [2]:
# Import helper classes 
from dataprep import CLASSES, get_train_transforms, get_val_transforms
from resnet18.terraria_data import TerrariaBiomeDataset, create_dataloaders

# CSV and image paths
CSV_PATH = os.path.join(PROJECT_ROOT, "data", "captures.csv")
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
from torch.utils.data import random_split
full_dataset = TerrariaBiomeDataset(
    csv_path=CSV_PATH,
    root_dir=DATA_ROOT,
    transform=get_train_transforms(),
    crop_size=(216,384),
    crops_per_image=9
)

# 80/20 train/val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

In [5]:
# set different transforms for train and val
train_dataset.dataset.transform = get_train_transforms()
val_dataset.dataset.transform = get_val_transforms()

In [6]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [7]:
print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

img, label = next(iter(train_loader))
print("Batch shape:", img.shape)
print("Batch labels:", label)

Train dataset size: 10922
Validation dataset size: 2731
Batch shape: torch.Size([32, 3, 216, 384])
Batch labels: tensor([12,  6,  9,  3, 10,  5, 10,  5,  6,  5,  1,  7,  9,  4, 12, 12,  4,  7,
         9,  7,  0,  3, 12,  8, 11,  1,  0, 11,  4, 12,  5,  7])


In [8]:
from resnet18.resnet_model import get_resnet18

model = get_resnet18()
model = model.to(device)

In [9]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)



In [10]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=3, 
)

In [11]:
import pandas as pd
import torch.nn as nn

# get training labels
train_labels = [
    full_dataset.data.iloc[i // full_dataset.crops_per_image]['biome']
    for i in train_dataset.indices
]

# compute counts per class
counts = pd.Series(train_labels).value_counts().sort_index()

print("Training counts per class:")
print(counts)

# inverse frequency weights
weights = 1.0 / counts
weights = weights / weights.sum()

class_weights = torch.tensor(
    [weights[c] for c in counts.index],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

Training counts per class:
Corruption      687
Crimson         868
Desert          819
Dungeon         924
Forest         1073
Hallow          554
Hell            988
Jungle         1135
Mushroom        478
Ocean           522
Snow            832
Space           542
Underground    1500
Name: count, dtype: int64


In [12]:
import time
import copy

EPOCHS = 10
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")

    # ---------- Training ----------
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        running_corrects += (outputs.argmax(1) == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = running_corrects / len(train_loader.dataset)
    print(f"Train Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

    # ---------- Validation ----------
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_corrects += (outputs.argmax(1) == labels).sum().item()

    val_loss = val_loss / len(val_loader.dataset)
    val_acc = val_corrects / len(val_loader.dataset)
    print(f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    # scheduler step
    if scheduler:
        scheduler.step(val_loss)

    # save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, "checkpoints/best_resnet18.pth")
        print("Saved best model!")

Epoch 1/10
Train Loss: 0.3944, Acc: 0.8922
Val Loss: 0.1232, Acc: 0.9623
Saved best model!
Epoch 2/10
Train Loss: 0.1389, Acc: 0.9598
Val Loss: 0.1164, Acc: 0.9674
Saved best model!
Epoch 3/10
Train Loss: 0.1019, Acc: 0.9685
Val Loss: 0.0885, Acc: 0.9722
Saved best model!
Epoch 4/10
Train Loss: 0.0839, Acc: 0.9746
Val Loss: 0.0710, Acc: 0.9740
Saved best model!
Epoch 5/10
Train Loss: 0.0928, Acc: 0.9692
Val Loss: 0.0631, Acc: 0.9791
Saved best model!
Epoch 6/10
Train Loss: 0.0823, Acc: 0.9761
Val Loss: 0.0593, Acc: 0.9795
Saved best model!
Epoch 7/10
Train Loss: 0.0670, Acc: 0.9786
Val Loss: 0.0711, Acc: 0.9725
Epoch 8/10
Train Loss: 0.0769, Acc: 0.9755
Val Loss: 0.0595, Acc: 0.9802
Saved best model!
Epoch 9/10
Train Loss: 0.0560, Acc: 0.9807
Val Loss: 0.0351, Acc: 0.9872
Saved best model!
Epoch 10/10
Train Loss: 0.0589, Acc: 0.9810
Val Loss: 0.0574, Acc: 0.9810


In [13]:
model.load_state_dict(best_model_wts)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [14]:
import torch
dummy_input = torch.randn(1, 3, 216, 384).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "checkpoints/resnet18_terraria.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17
)